Read Strava data

In [6]:
# === PROJECT ROOT SETUP (DO NOT REMOVE) ===
import sys
from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_ROOT = Path(r"C:\Users\elois\OneDrive\Desktop\vitality-behaviour-analysis")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

repo_root = PROJECT_ROOT  # ✅ define once, use everywhere

print("Project root:", repo_root)

Project root: C:\Users\elois\OneDrive\Desktop\vitality-behaviour-analysis


In [7]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

repo_root = Path(r"C:\Users\elois\OneDrive\Desktop\vitality-behaviour-analysis")
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.points_policy import PointsPolicy

policy = PointsPolicy(weekly_goal_points=900)

In [8]:
RAW_DIR = repo_root / "data" / "activities"
PROCESSED_DIR = repo_root / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

CACHE_ACTIVITIES = PROCESSED_DIR / "activities.parquet"

print("RAW:", RAW_DIR)
print("CACHE:", CACHE_ACTIVITIES)

RAW: C:\Users\elois\OneDrive\Desktop\vitality-behaviour-analysis\data\activities
CACHE: C:\Users\elois\OneDrive\Desktop\vitality-behaviour-analysis\data\processed\activities.parquet


In [9]:
if CACHE_ACTIVITIES.exists():
    activities_df = pd.read_parquet(CACHE_ACTIVITIES)
    print("Loaded cached activities")
else:
    activities_df = build_activities_df(RAW_DIR)
    activities_df.to_parquet(CACHE_ACTIVITIES)
    print("Parsed and cached activities")

# ✅ enforce chronological order
activities_df["start_time"] = pd.to_datetime(activities_df["start_time"], utc=True)
activities_df = activities_df.sort_values("start_time").reset_index(drop=True)

display(activities_df.head())
display(activities_df.tail())

Parsed and cached activities


,start_time,avg_hr_bpm,duration_min,age,source,path
0,2014-12-25 03:30:06+00:00,NaN,39.966667,23,gpx,C:\Users\elois\OneDrive\Desktop\vitality-behav...
1,2014-12-26 03:24:44+00:00,NaN,47.316667,23,gpx,C:\Users\elois\OneDrive\Desktop\vitality-behav...
2,2014-12-27 12:57:01+00:00,NaN,20.883333,23,gpx,C:\Users\elois\OneDrive\Desktop\vitality-behav...
3,2014-12-28 04:22:33+00:00,NaN,107.816667,23,gpx,C:\Users\elois\OneDrive\Desktop\vitality-behav...
4,2014-12-28 16:00:15+00:00,NaN,19.766667,23,gpx,C:\Users\elois\OneDrive\Desktop\vitality-behav...


,start_time,avg_hr_bpm,duration_min,age,source,path
1131,2026-03-06 14:58:29+00:00,108.074188,153.811317,23,fit,C:\Users\elois\OneDrive\Desktop\vitality-behav...
1132,2026-03-08 04:02:11+00:00,167.851247,100.925133,23,fit,C:\Users\elois\OneDrive\Desktop\vitality-behav...
1133,2026-03-09 14:02:46+00:00,81.775791,62.826117,23,fit,C:\Users\elois\OneDrive\Desktop\vitality-behav...
1134,2026-03-10 13:44:28+00:00,137.958748,62.706200,23,fit,C:\Users\elois\OneDrive\Desktop\vitality-behav...
1135,2026-03-11 15:12:18+00:00,91.491429,60.297700,23,fit,C:\Users\elois\OneDrive\Desktop\vitality-behav...


In [11]:
def build_daily_points_df(activities_df: pd.DataFrame, policy: PointsPolicy) -> pd.DataFrame:
    df = activities_df.copy()
    ts = pd.to_datetime(df["start_time"], utc=True).dt.tz_convert(None)
    df["date"] = ts.dt.date
    df["date"] = pd.to_datetime(df["date"])

    daily = (
        df.groupby("date", as_index=False)
          .apply(lambda g: policy.daily_points(g))
          .rename(columns={None: "daily_points"})
    )
    daily["daily_points"] = daily["daily_points"].astype(int)
    return daily

daily_points_df = build_daily_points_df(activities_df, policy)
display(daily_points_df.head())

,date,daily_points
0,2014-12-25,0
1,2014-12-26,0
2,2014-12-27,0
3,2014-12-28,0
4,2014-12-30,0


In [13]:
daily_points_df = build_daily_points_df(activities_df, policy)

DAILY_CACHE = PROCESSED_DIR / "daily_points.parquet"
daily_points_df.to_parquet(DAILY_CACHE)

print("Saved:", DAILY_CACHE)
display(daily_points_df.head(10))
display(daily_points_df.tail(10))

Saved: C:\Users\elois\OneDrive\Desktop\vitality-behaviour-analysis\data\processed\daily_points.parquet


,date,daily_points
0,2014-12-25,0
1,2014-12-26,0
2,2014-12-27,0
3,2014-12-28,0
4,2014-12-30,0
5,2015-01-03,0
6,2015-01-07,0
7,2015-01-10,0
8,2015-01-27,0
9,2015-02-01,0


,date,daily_points
1076,2026-02-27,0
1077,2026-02-28,300
1078,2026-03-02,200
1079,2026-03-03,100
1080,2026-03-04,0
1081,2026-03-06,0
1082,2026-03-08,300
1083,2026-03-09,0
1084,2026-03-10,300
1085,2026-03-11,0


In [14]:
print("Date range:")
print(daily_points_df["date"].min(), "→", daily_points_df["date"].max())

print("Non‑zero days:", (daily_points_df["daily_points"] > 0).sum())

Date range:
2014-12-25 00:00:00 → 2026-03-11 00:00:00
Non‑zero days: 439
